# Ship-motion benchmark on Kaggle

Run this notebook with **Accelerator: GPU**. It clones the public repository, retains the supplied raw CSVs and immutable M0--M2 manifests, then runs the leakage tests and the M3--M7 benchmark. Generated results are written under `/kaggle/working`.

The M8--M12 cell is deliberately opt-in: it is longer-running and should be used only after the primary benchmark has completed and its artifacts are frozen.

In [ ]:
from pathlib import Path
import os
import subprocess
import sys

REPOSITORY_URL = 'https://github.com/kushc2004/ship-motion-reimplementation.git'
WORKDIR = Path('/kaggle/working')
PROJECT_ROOT = WORKDIR / 'ship-motion-reimplementation'

if not PROJECT_ROOT.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPOSITORY_URL, str(PROJECT_ROOT)], check=True)
os.chdir(PROJECT_ROOT)
print('Project root:', Path.cwd())
print('Python:', sys.version.split()[0])
!nvidia-smi || true

In [ ]:
# Kaggle images normally include these packages. Install missing dependencies
# from the repository list; enable Internet in Kaggle only if installation is needed.
!{sys.executable} -m pip install -q -r requirements-kaggle.txt

import torch
assert torch.cuda.is_available(), 'Enable a Kaggle GPU accelerator before running neural models.'
print('CUDA device:', torch.cuda.get_device_name(0))

In [ ]:
# Confirm that the exact supplied sources and immutable artifacts are present.
required = [
    Path('BTP-1/Data.csv'),
    Path('BTP-1/Ship-Parameters-data.csv'),
    Path('BTP-2/BTP_2_data - Sheet1.csv'),
    Path('data/manifests/run_manifest.parquet'),
    Path('data/splits/BTP-1_windows.parquet'),
    Path('data/splits/BTP-2_windows.parquet'),
]
missing = [str(path) for path in required if not path.exists()]
if missing:
    raise FileNotFoundError(f'Missing repository inputs: {missing}')
print('Immutable M0--M2 inputs are present.')

In [ ]:
# Run the hard leakage and boundary tests before training.
!pytest -q tests/test_m02.py tests/test_m37.py

In [ ]:
# M3--M7: persistence, linear trend, Ridge, RF, XGBoost, LSTM, Transformer.
# All use the immutable manifests and the central evaluator.
!python scripts/run_primary_benchmark.py

In [ ]:
# Inspect the primary frozen-test comparison after the previous cell completes.
import pandas as pd
comparison_path = Path('outputs/primary_benchmark_m37/primary_comparison.csv')
display(pd.read_csv(comparison_path))

## Optional: M8--M12

Only run this after the preceding primary output has been reviewed and frozen. It consumes the existing immutable manifests and does not rebuild them. M12 export remains gated on completion of the required M10 and M11 artifacts.

In [ ]:
# Optional long-running later-phase experiments.
# !python scripts/run_m8_m12.py
# !pytest -q tests/test_m812.py
# !python scripts/export_final_report.py
# !python scripts/export_cv_claims.py